In [36]:
#TO EXTRACT FOLDER IN GOOGLE COLAB
!unrar x /content/ml_doraemon.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/ml_doraemon.rar

Creating    machin_learning_doraemon                                  OK
Creating    machin_learning_doraemon/test                             OK
Creating    machin_learning_doraemon/test/doraemon                    OK
Extracting  machin_learning_doraemon/test/doraemon/download.webp           1%  OK 
Extracting  machin_learning_doraemon/test/doraemon/gzj82tpo0cmf1.jpeg       1%  OK 
Extracting  machin_learning_doraemon/test/doraemon/images (1).jpg          2%  OK 
Extracting  machin_learning_doraemon/test/doraemon/images (2).jpg          2%  OK 
Extracting  machin_learning_doraemon/test/doraemon/images (3).jpg          2%  OK 
Extracting  machin_learning_doraemon/test/doraemon/images (5).jpg          2%  OK 
Extracting  machin_learning_doraemon/test/doraemon/images.jpg              2%  OK 
Creating    machin

In [37]:
import torch
import torchvision
from pathlib import Path
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader

In [38]:
dataset=Path("/content/machin_learning_doraemon")
train_path=dataset/"train"
test_path=dataset/"test"
val_path=dataset/"valid"
print("DATASET PATH LOADED")

DATASET PATH LOADED


In [39]:
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
print("DATASET TRANSFORMATION")

DATASET TRANSFORMATION


In [41]:
train_dataset=ImageFolder(train_path,transform=train_transform)
test_dataset=ImageFolder(test_path,transform=transform)
# val_dataset=ImageFolder(val_path,transform=transform)
print("DATASET LOADED TO IMAGEFOLDER")
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)
# val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)
print("DATASET LOADED TO DATALOADER")

DATASET LOADED TO IMAGEFOLDER
DATASET LOADED TO DATALOADER


In [42]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
weight=ResNet50_Weights
model=resnet50(weights=weight.DEFAULT)
model.fc=nn.Linear(2048,5)
criterion=nn.CrossEntropyLoss()
print("MODEL LOADED")

MODEL LOADED


In [43]:
for parameter in model.parameters():
  parameter.requires_grad=False
for parameter in model.fc.parameters():
  parameter.requires_grad=True
optimizer=torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)
print("OPTIMIZER TUNED")

OPTIMIZER TUNED


In [49]:

import copy
num=30
best_loss=float("inf")
best_model=None
for epoch in range(num):
  train_loss=0
  model.train()
  for inputs, labels in train_loader:
    optimizer.zero_grad()
    output=model(inputs)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()
  train_epoch_loss=train_loss/len(train_loader)
  print("-----------------")
  print(f"EPOCH :{epoch}")
  print(f"EPOCH TRAIN LOSS : {train_epoch_loss}")
  model.eval()
  with torch.no_grad():
   val_loss=0
   for inputs, labels in test_loader:
    output=model(inputs)
    loss=criterion(output,labels)
    val_loss+=loss.item()
   val_epoch_loss=val_loss/len(test_loader)
   if val_epoch_loss < best_loss:
    best_loss=val_epoch_loss
    best_model=copy.deepcopy(model.state_dict())
   print(f"VALIDATION LOSS : {val_epoch_loss} ,LOWEST VAL LOSS: {best_loss} ")


-----------------
EPOCH :0
EPOCH TRAIN LOSS : 0.2415337324142456
VALIDATION LOSS : 0.5716879814863205 ,LOWEST VAL LOSS: 0.5532912164926529 
-----------------
EPOCH :1
EPOCH TRAIN LOSS : 0.25479628443717955
VALIDATION LOSS : 0.5359376966953278 ,LOWEST VAL LOSS: 0.5359376966953278 
-----------------
EPOCH :2
EPOCH TRAIN LOSS : 0.23013209402561188
VALIDATION LOSS : 0.5266882926225662 ,LOWEST VAL LOSS: 0.5266882926225662 
-----------------
EPOCH :3
EPOCH TRAIN LOSS : 0.2132144331932068
VALIDATION LOSS : 0.5428676158189774 ,LOWEST VAL LOSS: 0.5266882926225662 
-----------------
EPOCH :4
EPOCH TRAIN LOSS : 0.22080269753932952
VALIDATION LOSS : 0.543449267745018 ,LOWEST VAL LOSS: 0.5266882926225662 
-----------------
EPOCH :5
EPOCH TRAIN LOSS : 0.1897028475999832
VALIDATION LOSS : 0.5203885287046432 ,LOWEST VAL LOSS: 0.5203885287046432 
-----------------
EPOCH :6
EPOCH TRAIN LOSS : 0.2237946003675461
VALIDATION LOSS : 0.5094858855009079 ,LOWEST VAL LOSS: 0.5094858855009079 
-----------------


In [50]:
torch.save(best_model, "train3Doraemon.pth")

In [51]:
# Print the index-to-class mapping created by ImageFolder
print("Class to Index Mapping:")
print(train_dataset.class_to_idx)

Class to Index Mapping:
{'doraemon': 0, 'gian': 1, 'nobita': 2, 'shizuka': 3, 'suneo': 4}


In [55]:
#MANUAL TESTING FOR SINGLE IMAGE
from PIL import Image
import torch

image_path = "/content/shizuka.webp"

image = Image.open(image_path).convert("RGB")

image = transform(image)

image = image.unsqueeze(0)

model.eval()

with torch.no_grad():
    output = model(image)

    probabilities = torch.softmax(output, dim=1)

    confidence, prediction = torch.max(probabilities, dim=1)

class_names = [
    "DORAEMON",
    "GIAN",
    "NOBITA",
    "SHIZUKA",
    "SUNEO"
]

print("Prediction:", class_names[prediction.item()])
print("Confidence:", confidence.item() * 100, "%")

Prediction: SHIZUKA
Confidence: 37.85023391246796 %
